# Multi-Cancer Detection System
This notebook contains the complete pipeline for handling multiple cancer datasets (Breast Cancer and Cervical Cancer), including data preprocessing, model training, dynamic routing, and evaluation.

In [11]:
import os
import urllib.request
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

ModuleNotFoundError: No module named 'numpy'

## 1. Data Preprocessing
Functions to load, clean, scale, and split our datasets.

In [ ]:
def load_and_preprocess_breast_cancer():
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, data.feature_names, data.target_names

def load_and_preprocess_cervical_cancer():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00383/risk_factors_cervical_cancer.csv'
    dataset_path = 'cervical_cancer.csv'
    
    if not os.path.exists(dataset_path):
        print('Downloading Cervical Cancer dataset...')
        urllib.request.urlretrieve(url, dataset_path)
        
    df = pd.read_csv(dataset_path, na_values='?')
    df = df.drop(columns=['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis'])
    
    target_col = 'Biopsy'
    df = df.drop(columns=['Hinselmann', 'Schiller', 'Citology'])
    
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
            
    X = df.drop(columns=[target_col])
    y = df[target_col].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    target_names = np.array(['Healthy', 'Cancer'])
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, X.columns, target_names

def get_dataset(cancer_type='breast'):
    if cancer_type == 'breast':
        return load_and_preprocess_breast_cancer()
    elif cancer_type == 'cervical':
        return load_and_preprocess_cervical_cancer()
    else:
        raise ValueError(f'Unknown cancer type: {cancer_type}')

## 2. Model Training
Trains ensemble models (Random Forest, SVM, Gradient Boosting) for both datasets and saves them using Pickle.

In [ ]:
def train_ensemble(X_train, y_train, X_test, y_test, cancer_type):
    print(f'--- Training models for {cancer_type} cancer ---')
    
    rf = RandomForestClassifier(random_state=42)
    svm = SVC(probability=True, random_state=42)
    gb = GradientBoostingClassifier(random_state=42)
    
    rf.fit(X_train, y_train)
    svm.fit(X_train, y_train)
    gb.fit(X_train, y_train)
    
    ensemble = VotingClassifier(estimators=[('rf', rf), ('svm', svm), ('gb', gb)], voting='soft')
    ensemble.fit(X_train, y_train)
    
    print(f'Ensemble Accuracy: {accuracy_score(y_test, ensemble.predict(X_test)):.4f}\n')
    return ensemble

def train_and_save_all_models():
    cancer_types = {
        'breast': ('cancer_model.pkl', 'scaler.pkl'),
        'cervical': ('cervical_model.pkl', 'cervical_scaler.pkl')
    }
    
    for ctype, (model_filename, scaler_filename) in cancer_types.items():
        X_train, X_test, y_train, y_test, scaler, _, _ = get_dataset(ctype)
        ensemble_model = train_ensemble(X_train, y_train, X_test, y_test, ctype)
        
        with open(model_filename, 'wb') as f:
            pickle.dump(ensemble_model, f)
            
        with open(scaler_filename, 'wb') as f:
            pickle.dump(scaler, f)
            
        print(f'Saved {ctype} model to {model_filename}')

In [ ]:
train_and_save_all_models()

## 3. Dynamic Prediction Router
A class to dynamically load the appropriate model and format predictions.

In [ ]:
class CancerPredictionRouter:
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.target_names = {
            'breast': ['Malignant', 'Benign'],
            'cervical': ['Healthy', 'Cancer']
        }
        self._load_models()

    def _load_models(self):
        cancer_types = {
            'breast': ('cancer_model.pkl', 'scaler.pkl'),
            'cervical': ('cervical_model.pkl', 'cervical_scaler.pkl')
        }
        for ctype, (model_file, scaler_file) in cancer_types.items():
            if os.path.exists(model_file) and os.path.exists(scaler_file):
                with open(model_file, 'rb') as f:
                    self.models[ctype] = pickle.load(f)
                with open(scaler_file, 'rb') as f:
                    self.scalers[ctype] = pickle.load(f)

    def predict(self, cancer_type, features):
        scaler = self.scalers[cancer_type]
        model = self.models[cancer_type]
        
        features_array = np.array(features).reshape(1, -1)
        scaled_features = scaler.transform(features_array)
        
        prediction = model.predict(scaled_features)[0]
        probabilities = model.predict_proba(scaled_features)[0]
        
        risk_score = probabilities[0] if cancer_type == 'breast' else probabilities[1]
            
        return {
            'cancer_type': cancer_type,
            'prediction': int(prediction),
            'diagnosis': self.target_names[cancer_type][prediction],
            'risk_score': float(risk_score)
        }

In [ ]:
router = CancerPredictionRouter()

breast_dummy = np.zeros(30)
print('Breast Cancer Sample:', router.predict('breast', breast_dummy))

cervical_dummy = np.zeros(30)
print('Cervical Cancer Sample:', router.predict('cervical', cervical_dummy))

## 4. Evaluation
Evaluates the models and generates a comprehensive report.

In [ ]:
def evaluate_models():
    cancer_types = ['breast', 'cervical']
    router = CancerPredictionRouter()
    evaluation_results = []
    
    for ctype in cancer_types:
        print(f'\n==== Evaluating {ctype.capitalize()} Cancer Model ====')
        _, X_test_scaled, _, y_test, _, _, target_names = get_dataset(ctype)
        model = router.models[ctype]
        
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan
            
        print(f'Accuracy:  {acc:.4f}')
        print(f'Precision: {prec:.4f}')
        print(f'Recall:    {rec:.4f}')
        print(f'F1-Score:  {f1:.4f}')
        if not np.isnan(auc): print(f'ROC-AUC:   {auc:.4f}')
            
        print('\nClassification Report:')
        print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))
        
        evaluation_results.append({
            'Cancer Type': ctype.capitalize(),
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': auc
        })
        
    summary_df = pd.DataFrame(evaluation_results)
    print('\n==== Summary of All Models ====')
    print(summary_df.to_string(index=False))

evaluate_models()